Changing the stage type, and DIRKs
==================================

By default, Irksome solves the classical Runge--Kutta formulation where the stage variables 
approximate the function derivatives.  It is also possible to pose the method in terms
of stage variables representing the solution itself.

For ODE, the RK formulation is
$$
Y_i = F\left(t^n + c_i \Delta t, y^n + \sum_{j=1}^s A_{ij} Y_j\right)
$$
and we update the solution by
$$
y^{n+1} = y^n + \sum_{i=1}^s b_i F\left(t^n + c_i \Delta t, Y_i\right).
$$
Two notes are in order:
- in FEM, the update must be carried out in weak form, solving a mass matrix.
- For the important class stiffly accurate methods (RaduaIIA, LobattoIIIC), maths gives us that $y^{n+1} = Y_s$, so this extra arithmetic is not needed

We continue with heat equation demonstration problem on $\Omega = [0,10]
\times [0,10]$, with boundary $\Gamma$, with weak form as in earlier demos.



In [44]:
from firedrake import *  # noqa: F403
from irksome import RadauIIA, TimeStepper, Dt

Here is our setup as in the previous demos:

In [45]:
butcher_tableau = RadauIIA(2)
N = 64
x0 = 0.0
x1 = 10.0
y0 = 0.0
y1 = 10.0
msh = RectangleMesh(N, N, x1, y1)
dt = Constant(10. / N)
t = Constant(0.0)
V = FunctionSpace(msh, "CG", 1)
x, y = SpatialCoordinate(msh)
S = Constant(2.0)
C = Constant(1000.0)
B = (x-Constant(x0))*(x-Constant(x1))*(y-Constant(y0))*(y-Constant(y1))/C
R = (x * x + y * y) ** 0.5
uexact = B * atan(t)*(pi / 2.0 - atan(S * (R - t)))
rhs = Dt(uexact) - div(grad(uexact))
u = Function(V)
u.interpolate(uexact)
v = TestFunction(V)
F = inner(Dt(u), v)*dx + inner(grad(u), grad(v))*dx - inner(rhs, v)*dx
bc = DirichletBC(V, 0, "on_boundary")

Rana works about the same in this form (sometimes with slightly different iteration counts)

In [46]:
ranaparams = {
    "mat_type": "aij",
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "ksp_monitor": None,
    "pc_type": "python",
    "pc_python_type": "irksome.RanaLD",
    "aux": {
        "pc_type": "fieldsplit",
        "pc_fieldsplit_type": "multiplicative",
        "fieldsplit": {
            "ksp_type": "preonly",
            "pc_type": "gamg"
        }
    }
}



And we access the stage value form by a keyword argument.

In [47]:
stepper = TimeStepper(F, butcher_tableau, t, dt, u, bcs=bc, stage_type="value",
                      solver_parameters=ranaparams)
stepper.advance()

    Residual norms for firedrake_22_ solve.
    0 KSP Residual norm 2.739810230564e-01
    1 KSP Residual norm 4.327695990306e-03
    2 KSP Residual norm 4.148999727186e-04
    3 KSP Residual norm 7.468808468418e-05
    4 KSP Residual norm 1.845502319180e-05
    5 KSP Residual norm 3.674166002538e-06
    6 KSP Residual norm 7.699147084709e-07


Diagonally implicit Runge--Kutta schemes
========================================

*Diagonally implicit* methods are a commonly used technique to avoid having to solve a stage-coupled system, instead solving for the stages in sequence.  There is a classical L-stable 3-stage DIRK due to Alexander (SINUM, 1977):

In [48]:
from irksome import Alexander
btalex = Alexander()
btalex.A

array([[ 0.43586652,  0.        ,  0.        ],
       [ 0.28206674,  0.43586652,  0.        ],
       [ 1.20849665, -0.64436317,  0.43586652]])

Because the Butcher matrix is lower triangular, we can actually solve the stages in sequence with any solver suitable for backward Euler.  This requires an extra keyward argument to `TimeStepper`:


In [49]:
gamg_params = {
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "pc_type": "gamg"
}
stepper = TimeStepper(F, btalex, t, dt, u, bcs=bc, stage_type="dirk",
                      solver_parameters=gamg_params)

And we can integrate the system:

In [50]:
while (float(t) < 1.0):
    if (float(t) + float(dt) > 1.0):
        dt.assign(1.0 - t)
    stepper.advance()
    print(float(t))
    t.assign(t + dt)
print()
print(errornorm(uexact, u)/norm(uexact))

0.0
0.15625
0.3125
0.46875
0.625
0.78125
0.9375

0.1270061943013998


However, note that Irksome makes the fully implicit methods about as easy to apply as diagonally implicit ones, and with good solvers, you have an efficient and accurate method without order reduction that can occur in DIRKS

In [51]:
t.assign(0)
dt.assign(10/N)
u.interpolate(uexact)

while (float(t) < 1.0):
    if (float(t) + float(dt) > 1.0):
        dt.assign(1.0 - t)
    stepper.advance()
    print(float(t))
    t.assign(t + dt)
print()
print(errornorm(uexact, u)/norm(uexact))


0.0
0.15625
0.3125
0.46875
0.625
0.78125
0.9375

0.0024338205181291066


On the other hand, if we integrated with RaduaIIA(2), which is also formally third order, we get a slightly better answer.
It's not staggering here, but 

In [ ]:
t.assign(0)
dt.assign(10/N)
u.interpolate(uexact)
ranaparams.pop("ksp_monitor")  # less output!
stepper = TimeStepper(F, RadauIIA(2), t, dt, u, bcs=bc,
                      solver_parameters=ranaparams)
while (float(t) < 1.0):
    if (float(t) + float(dt) > 1.0):
        dt.assign(1.0 - t)
    stepper.advance()
    print(float(t))
    t.assign(t + dt)
print()
print(errornorm(uexact, u)/norm(uexact))


0.0
0.15625
0.3125
0.46875
0.625
0.78125
0.9375

0.0024056144842234295
